> ### ⚠️ These results contain look-ahead bias — do not quote them
>
> The Sharpe ratio reported below (**8.59** net of costs) is an artifact of
> this notebook's construction, not a research result. The strategy trades the
> cumulative sum of the Kalman filter's own one-step innovations, and the
> filter's update step manufactures the negative autocorrelation it then
> trades: sweeping the process-noise parameter moves Sharpe from 1.93 to 9.03.
> The OU calibration and the universe selection are also fitted on the full
> sample.
>
> This notebook is kept as the record of the original construction.
> **For the corrected, look-ahead-free pipeline see
> `05_causal_backtest_validation.ipynb`**, which reports Sharpe **0.52**
> (sign-shuffle null p = 0.110, i.e. not statistically significant).
> The full analysis is in `results/BACKTEST_FINDINGS.md`.

# TCN VaR Risk Overlay

Trains a Temporal Convolutional Network to forecast portfolio tail risk (1% and 5% VaR) via pinball loss, validates the forecasts with the Kupiec POF test, and applies a dynamic de-leveraging overlay to the base strategy.

## 1: Environment Setup & Module Imports

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from src.backtest import PortfolioBacktester
from src.clustering import compute_clusters_isolated
from src.data_loader import fetch_equity_returns
from src.diagnostics import SpreadDiagnostics
from src.ou_process import OUProcessModel
from src.residuals_KF import extract_idiosyncratic_residuals
from src.rolling_engine import RollingPCAEngine
from src.tcn_var import (
    Chomp1d,
    PinballLoss,
    PortfolioSequenceDataset,
    TCNVaRForecaster,
    TemporalBlock,
    apply_var_risk_overlay,
    kupiec_pof_test,
    prepare_stat_arb_features,
)

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

plt.style.use(
    "seaborn-v0_8-whitegrid"
    if "seaborn-v0_8-whitegrid" in plt.style.available
    else "default"
)
plt.rcParams["figure.figsize"] = (12, 6)

## 2: Upstream Strategy Pipeline Execution (2-Year Horizon)

In [ ]:
tickers = [
    "AAPL",
    "MSFT",
    "GOOGL",
    "AMZN",
    "NVDA",
    "META",
    "TSLA",
    "AMD",
    "INTC",
    "QCOM",
    "JPM",
    "BAC",
    "WFC",
    "C",
    "GS",
    "MS",
    "XOM",
    "CVX",
    "COP",
    "SLB",
]
start_date = "2019-01-01"  # earlier start gives room to roll a 2yr window
end_date = "2025-01-01"
backtest_start = "2023-01-01"

# 1. Ingestion & Walk-Forward PCA Factor Extraction
returns = fetch_equity_returns(
    tickers, start_date=start_date, end_date=end_date
)
rolling_pca = RollingPCAEngine(n_components=5, window=504, rebalance_freq=21)
factor_returns = rolling_pca.fit_transform(returns)

returns = returns.loc[backtest_start:]
factor_returns = factor_returns.loc[backtest_start:]

# 2. Dynamic Asset Clustering (latest loadings snapshot as of backtest start)
# Clustering runs in a separate process: UMAP pulls in TensorFlow, which
# segfaults when it shares a process with the PyTorch TCN trained below.
clusters = compute_clusters_isolated(
    rolling_pca.loadings_as_of(returns.index[0]),
    n_components=2,
    eps=0.4,
    min_samples=2,
)
noise_assets = set(clusters.get(-1, []))

# 3. Kalman Filter Residual Extraction
residuals, cumulative_spreads, _ = extract_idiosyncratic_residuals(
    returns, factor_returns, process_noise=1e-4, measurement_noise=1e-3
)

burn_in = 30
clean_residuals = residuals.iloc[burn_in:]
clean_spreads = cumulative_spreads.iloc[burn_in:]

# 4. Statistical Filtering & OU Calibration
diag = SpreadDiagnostics(significance_level=0.05)
diagnostic_summary = diag.filter_tradeable_spreads(
    clean_spreads, clean_residuals
)
tradeable_mask = diagnostic_summary["tradeable"] & (
    ~diagnostic_summary.index.isin(noise_assets)
)
active_universe = diagnostic_summary[tradeable_mask].index.tolist()

ou_engine = OUProcessModel()
s_scores = pd.DataFrame(index=clean_spreads.index, columns=active_universe)
sigma_eq_dict = {}

for ticker in active_universe:
    params = ou_engine.fit_spread(clean_spreads[ticker])
    if not np.isnan(params["sigma_eq"]):
        sigma_eq_dict[ticker] = params["sigma_eq"]
        s_scores[ticker] = ou_engine.compute_s_score(clean_spreads[ticker])

s_scores = s_scores.dropna(how="all", axis=1).astype(float)

# 5. Base Signal Generation & Unhedged Baseline Execution
backtester = PortfolioBacktester(
    s_open=1.25, s_close=0.5, transaction_cost_bps=5.0
)
raw_signals = backtester.generate_signals(s_scores)
base_weights = backtester.compute_portfolio_weights(
    raw_signals, sigma_eq_dict
)
base_results = backtester.run_backtest(
    base_weights, clean_residuals[s_scores.columns]
)

print(
    f"Upstream pipeline completed across {len(active_universe)} tradeable assets."
)

## 3: 3D Feature Tensor Preparation & Dataset Splitting

In [ ]:
# Align systematic macro PCA factors to the cleaned date index
clean_factors = factor_returns.loc[clean_spreads.index]

# Build multi-channel feature matrix: [R_p, R_p^2, F_1 ... F_K]
feature_matrix, portfolio_returns = prepare_stat_arb_features(
    positions=base_weights,
    residuals=clean_residuals[s_scores.columns],
    factor_returns=clean_factors,
    sigma_eq_dict=sigma_eq_dict,
)

# Chronological Train/Test Split (70% Train, 30% Out-of-Sample Test)
seq_len = 30
split_idx = int(0.70 * len(feature_matrix))

train_dataset = PortfolioSequenceDataset(
    feature_matrix[:split_idx], portfolio_returns[:split_idx], seq_len=seq_len
)
test_dataset = PortfolioSequenceDataset(
    feature_matrix[split_idx:], portfolio_returns[split_idx:], seq_len=seq_len
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print(
    f"Feature Dimensions: {feature_matrix.shape[1]} channels (1 Return + 1 Vol Proxy + {clean_factors.shape[1]} PCA Factors)"
)
print(f"Training Windows:   {len(train_dataset)} tensors of shape (32, {feature_matrix.shape[1]}, {seq_len})")
print(f"Out-of-Sample Eval: {len(test_dataset)} tensors")

## 4: TCN Architecture Instantiation & Quantile Loss Training

In [ ]:
n_features = feature_matrix.shape[1]
quantiles = [0.01, 0.05]

tcn_model = TCNVaRForecaster(
    num_inputs=n_features,
    num_channels=[16, 32, 64],
    kernel_size=3,
    dropout=0.15,
    quantiles=quantiles,
)

criterion = PinballLoss(quantiles=quantiles)
optimizer = torch.optim.AdamW(
    tcn_model.parameters(), lr=0.003, weight_decay=1e-4
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=5
)

# Training Loop
epochs = 35
loss_history = []
tcn_model.train()

for epoch in range(1, epochs + 1):
    epoch_loss = 0.0
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        predictions = tcn_model(batch_x)
        loss = criterion(predictions, batch_y)
        loss.backward()
        # Gradient clipping prevents gradient explosion in deep causal layers
        torch.nn.utils.clip_grad_norm_(tcn_model.parameters(), max_norm=1.0)
        optimizer.step()
        epoch_loss += loss.item() * batch_x.size(0)

    avg_loss = epoch_loss / len(train_dataset)
    loss_history.append(avg_loss)
    scheduler.step(avg_loss)

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:02d}/{epochs} | Multi-Quantile Pinball Loss: {avg_loss:.6f}")

# Plot Training Convergence
plt.figure(figsize=(9, 4))
plt.plot(loss_history, color="navy", lw=2, label="Pinball Loss")
plt.title("TCN Multi-Quantile Loss Convergence across Epochs")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

## 5: Out-of-Sample Forecasting & Kupiec POF Statistical Validation

In [ ]:
tcn_model.eval()
test_preds, test_actuals = [], []

with torch.no_grad():
    for batch_x, batch_y in test_loader:
        preds = tcn_model(batch_x)
        test_preds.append(preds)
        test_actuals.append(batch_y)

var_preds = torch.cat(test_preds, dim=0).numpy()
actual_pnl = torch.cat(test_actuals, dim=0).numpy().flatten()

# Extract 1% and 5% VaR forecasts
var_1pct = var_preds[:, 0]
var_5pct = var_preds[:, 1]

# Econometric Validation: Kupiec POF Likelihood Ratio Tests
kupiec_1pct = kupiec_pof_test(actual_pnl, var_1pct, alpha=0.01)
kupiec_5pct = kupiec_pof_test(actual_pnl, var_5pct, alpha=0.05)

eval_summary = pd.DataFrame(
    [kupiec_1pct, kupiec_5pct], index=["1% VaR Forecast", "5% VaR Forecast"]
)
print("=== Kupiec POF Unconditional Coverage Test Results ===")
print(
    eval_summary[
        ["alpha", "failures", "failure_rate", "lr_stat", "p_value", "model_accepted"]
    ]
)

## 6: Visualizing Realized PnL vs. Predicted Dynamic VaR Envelopes

In [ ]:
test_dates = clean_spreads.index[split_idx + seq_len :]

plt.figure(figsize=(13, 6))
plt.plot(
    test_dates,
    actual_pnl,
    color="black",
    lw=1.2,
    alpha=0.75,
    label="Realized Portfolio Return",
)
plt.plot(
    test_dates,
    var_5pct,
    color="darkorange",
    ls="--",
    lw=1.8,
    label="TCN 5% Dynamic VaR",
)
plt.plot(
    test_dates,
    var_1pct,
    color="crimson",
    ls="-.",
    lw=1.8,
    label="TCN 1% Dynamic VaR (Extreme Tail)",
)

# Highlight VaR Breaches
breach_mask_5 = actual_pnl < var_5pct
plt.scatter(
    test_dates[breach_mask_5],
    actual_pnl[breach_mask_5],
    color="red",
    s=40,
    zorder=5,
    label="5% VaR Breach",
)

plt.title("Out-of-Sample Realized Portfolio Returns vs. TCN Tail VaR Forecasts")
plt.ylabel("Daily Strategy Return")
plt.xlabel("Date")
plt.legend(loc="lower left")
plt.tight_layout()
plt.show()

## 7: Dynamic Risk Overlay & Comparative Performance Execution

In [ ]:
# Generate full-sample VaR forecasts for complete overlay execution
full_dataset = PortfolioSequenceDataset(
    feature_matrix, portfolio_returns, seq_len=seq_len
)
full_loader = DataLoader(full_dataset, batch_size=64, shuffle=False)

full_preds = []
with torch.no_grad():
    for batch_x, _ in full_loader:
        full_preds.append(tcn_model(batch_x))
full_var_forecasts = torch.cat(full_preds, dim=0).numpy()

# Apply Dynamic Sizing & Circuit Breaker Halts
# Base risk budget: maximum allowable 1% daily drawdown of 1.5%
hedged_weights = apply_var_risk_overlay(
    positions=base_weights,
    var_forecasts=full_var_forecasts,
    target_risk_limit=0.015,
    var_index_offset=seq_len,
)

# Run Backtests on Unhedged vs. Hedged Strategy
hedged_results = backtester.run_backtest(
    hedged_weights, clean_residuals[s_scores.columns]
)

# Comparative Tearsheet Metrics
base_metrics = backtester.calculate_metrics(base_results["net_returns"])
hedged_metrics = backtester.calculate_metrics(hedged_results["net_returns"])

comparison_df = pd.DataFrame(
    {"Base Strategy (Unhedged)": base_metrics, "TCN Risk Overlay": hedged_metrics}
)

print("=== Risk-Adjusted Performance Comparison Tearsheet ===")
print(comparison_df.round(4))

## 8: Final Visualizations — Equity Curve & Drawdown Defense

In [ ]:
fig, (ax1, ax2) = plt.subplots(
    2, 1, figsize=(13, 9), sharex=True, gridspec_kw={"height_ratios": [2.5, 1.5]}
)

# 1. Equity Curves
ax1.plot(
    base_results.index,
    base_results["net_equity"],
    color="gray",
    ls="--",
    lw=1.8,
    label="Base Strategy (Net of TC)",
)
ax1.plot(
    hedged_results.index,
    hedged_results["net_equity"],
    color="navy",
    lw=2.2,
    label="TCN Risk Overlay Strategy (Net of TC)",
)
ax1.set_ylabel("Portfolio Value (Base 1.0)")
ax1.set_title(
    "Tail-Risk Mitigation: Unhedged vs. TCN Causal VaR Overlay", fontsize=12
)
ax1.legend(loc="upper left")

# 2. Underwater Drawdown Comparison
dd_base = (
    base_results["net_equity"] - base_results["net_equity"].cummax()
) / base_results["net_equity"].cummax()
dd_hedged = (
    hedged_results["net_equity"] - hedged_results["net_equity"].cummax()
) / hedged_results["net_equity"].cummax()

ax2.plot(
    dd_base.index,
    dd_base,
    color="crimson",
    ls="--",
    lw=1.2,
    alpha=0.7,
    label="Base Strategy Drawdown",
)
ax2.fill_between(
    dd_hedged.index,
    dd_hedged,
    0,
    color="darkgreen",
    alpha=0.3,
    label="TCN Overlay Drawdown",
)
ax2.plot(dd_hedged.index, dd_hedged, color="darkgreen", lw=1.5)
ax2.set_ylabel("Drawdown")
ax2.set_xlabel("Date")
ax2.legend(loc="lower left")

plt.tight_layout()
plt.show()